In [1]:
import json
from pathlib import Path

import pandas as pd
import skrub

In [2]:
X_train = pd.read_csv("data/X_train.csv")
y_train = pd.read_csv("data/y_train.csv")
X_test = pd.read_csv("data/X_test.csv")

# Merge features + target into a single train table for analysis
RAW = X_train.merge(y_train, on="Index")

{"X_train_shape": X_train.shape, "y_train_shape": y_train.shape, "X_test_shape": X_test.shape, "merged_train_shape": RAW.shape}  # noqa: B018


{'X_train_shape': (44590, 13),
 'y_train_shape': (44590, 2),
 'X_test_shape': (11013, 13),
 'merged_train_shape': (44590, 14)}

In [3]:
RAW.head()

,Index,patient_id,cohort,sexM,gene,age_at_diagnosis,age,ledd,time_since_intake_on,time_since_intake_off,rater_id,on,off,target
0,0,IPLP5212,A,0,LRRK2+,48.5,52.1,607.0,1.9,NaN,R36,7.0,NaN,34.7
1,1,IPLP5212,A,0,LRRK2+,48.5,53.0,666.0,1.9,17.6,R43,11.0,40.0,38.1
2,2,IPLP5212,A,0,LRRK2+,48.5,53.9,717.0,1.2,NaN,R43,6.0,NaN,41.6
3,3,IPLP5212,A,0,LRRK2+,48.5,54.8,770.0,1.5,NaN,R36,12.0,NaN,44.9
4,4,IPLP5212,A,0,LRRK2+,48.5,56.9,885.0,0.3,NaN,R36,26.0,NaN,52.0


In [4]:
X_train

,Index,patient_id,cohort,sexM,gene,age_at_diagnosis,age,ledd,time_since_intake_on,time_since_intake_off,rater_id,on,off
0,0,IPLP5212,A,0,LRRK2+,48.5,52.1,607.0,1.9,NaN,R36,7.0,NaN
1,1,IPLP5212,A,0,LRRK2+,48.5,53.0,666.0,1.9,17.6,R43,11.0,40.0
2,2,IPLP5212,A,0,LRRK2+,48.5,53.9,717.0,1.2,NaN,R43,6.0,NaN
3,3,IPLP5212,A,0,LRRK2+,48.5,54.8,770.0,1.5,NaN,R36,12.0,NaN
4,4,IPLP5212,A,0,LRRK2+,48.5,56.9,885.0,0.3,NaN,R36,26.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
44585,55598,CZTX7061,A,1,LRRK2+,59.0,60.6,NaN,NaN,NaN,R49,NaN,1.0
44586,55599,CZTX7061,A,1,LRRK2+,59.0,61.1,NaN,NaN,NaN,R49,NaN,5.0
44587,55600,CZTX7061,A,1,LRRK2+,59.0,61.8,NaN,NaN,NaN,R39,NaN,0.0
44588,55601,CZTX7061,A,1,LRRK2+,59.0,62.6,NaN,NaN,NaN,R32,NaN,2.0


In [5]:
print(X_train["gene"][X_train["cohort"]=="A"].isna().sum()/X_train["gene"][X_train["cohort"]=="A"].shape[0]*100)

30.858310626702995


In [6]:
print(X_train["gene"][X_train["cohort"]=="B"].isna().sum()/X_train["gene"][X_train["cohort"]=="B"].shape[0]*100)

44.42874444893016


on ne remarque pas une différence énorme en pourcentage entre les différents cohort, pas forcément besoin de séparer le traitement pour les genes pour l'instant

In [7]:
df_intake = X_train.dropna(subset=["time_since_intake_on", "time_since_intake_off"])
df_intake.head()

,Index,patient_id,cohort,sexM,gene,age_at_diagnosis,age,ledd,time_since_intake_on,time_since_intake_off,rater_id,on,off
1,1,IPLP5212,A,0,LRRK2+,48.5,53.0,666.0,1.9,17.6,R43,11.0,40.0
7,7,IPLP5212,A,0,LRRK2+,48.5,59.9,888.0,0.2,9.6,R43,28.0,33.0
9,9,IPLP5212,A,0,LRRK2+,48.5,61.5,984.0,1.1,13.5,R36,12.0,46.0
10,10,IPLP5212,A,0,LRRK2+,48.5,63.3,1079.0,3.7,15.9,R43,13.0,50.0
18,18,ULEJ2538,A,1,NaN,71.6,74.5,513.0,0.8,12.2,R01,19.0,24.0


In [8]:
df_intake_2 = X_train.dropna(subset=["on", "off"])
df_intake_2.head()

,Index,patient_id,cohort,sexM,gene,age_at_diagnosis,age,ledd,time_since_intake_on,time_since_intake_off,rater_id,on,off
1,1,IPLP5212,A,0,LRRK2+,48.5,53.0,666.0,1.9,17.6,R43,11.0,40.0
6,6,IPLP5212,A,0,LRRK2+,48.5,58.9,835.0,NaN,NaN,R43,9.0,34.0
7,7,IPLP5212,A,0,LRRK2+,48.5,59.9,888.0,0.2,9.6,R43,28.0,33.0
9,9,IPLP5212,A,0,LRRK2+,48.5,61.5,984.0,1.1,13.5,R36,12.0,46.0
10,10,IPLP5212,A,0,LRRK2+,48.5,63.3,1079.0,3.7,15.9,R43,13.0,50.0


In [9]:
df_intake_3 = X_train.dropna(subset=["on", "off","time_since_intake_on", "time_since_intake_off"])
df_intake_3.head()

,Index,patient_id,cohort,sexM,gene,age_at_diagnosis,age,ledd,time_since_intake_on,time_since_intake_off,rater_id,on,off
1,1,IPLP5212,A,0,LRRK2+,48.5,53.0,666.0,1.9,17.6,R43,11.0,40.0
7,7,IPLP5212,A,0,LRRK2+,48.5,59.9,888.0,0.2,9.6,R43,28.0,33.0
9,9,IPLP5212,A,0,LRRK2+,48.5,61.5,984.0,1.1,13.5,R36,12.0,46.0
10,10,IPLP5212,A,0,LRRK2+,48.5,63.3,1079.0,3.7,15.9,R43,13.0,50.0
18,18,ULEJ2538,A,1,NaN,71.6,74.5,513.0,0.8,12.2,R01,19.0,24.0


In [10]:
X_train[X_train["patient_id"]=="CZTX7061"]

,Index,patient_id,cohort,sexM,gene,age_at_diagnosis,age,ledd,time_since_intake_on,time_since_intake_off,rater_id,on,off
44584,55597,CZTX7061,A,1,LRRK2+,59.0,59.7,NaN,NaN,NaN,R39,NaN,1.0
44585,55598,CZTX7061,A,1,LRRK2+,59.0,60.6,NaN,NaN,NaN,R49,NaN,1.0
44586,55599,CZTX7061,A,1,LRRK2+,59.0,61.1,NaN,NaN,NaN,R49,NaN,5.0
44587,55600,CZTX7061,A,1,LRRK2+,59.0,61.8,NaN,NaN,NaN,R39,NaN,0.0
44588,55601,CZTX7061,A,1,LRRK2+,59.0,62.6,NaN,NaN,NaN,R32,NaN,2.0
44589,55602,CZTX7061,A,1,LRRK2+,59.0,63.5,NaN,NaN,NaN,R32,NaN,8.0


gene → remplacer NaN par 'no_mutation'
✅ Bon choix, avec une nuance.

Le raisonnement clinique est solide : si le génotype n'est pas renseigné, c'est souvent parce qu'aucune mutation connue n'a été trouvée, ou que le test n'a pas été fait — dans les deux cas, la valeur 'no_mutation' capture l'absence d'information génétique spécifique. Avec seulement 4 valeurs uniques, c'est une colonne très basse cardinalité ; un encodage one-hot ou ordinal sera propre.

Nuance à surveiller : l'EDA montre que la missingness sur gene est probablement structurelle par cohorte (certains protocoles ne collectent pas le génotype). Si c'est le cas, 'no_mutation' et 'missing par protocole' sont deux choses différentes. Pour être précis, tu pourrais nommer la valeur 'unknown' plutôt que 'no_mutation' — mais l'impact pratique sur un arbre est nul, car l'arbre le traitera de toute façon comme une catégorie à part.

age_at_diagnosis → imputer par age - 5.7
⚠️ Acceptable comme heuristique, mais fragile.

L'idée est logique : si on ne connaît pas age_at_diagnosis, on l'estime à partir de l'âge actuel moins l'écart moyen observé. Le problème est que cette constante (5.7) est calculée sur la population globale, alors que l'écart réel varie beaucoup d'un patient à l'autre (maladie précoce vs tardive). Tu produis une valeur synthétique qui est corrélée à age par construction, ce qui n'apporte pas vraiment d'information nouvelle.

Alternative plus robuste : puisque age_at_diagnosis n'est manquant qu'à 5.2%, et que tu as plusieurs visites par patient, tu peux d'abord faire une propagation intra-patient : pour un patient dont age_at_diagnosis est manquant sur une visite mais présent sur d'autres, utilise la valeur de l'autre visite (c'est une constante par patient). Ensuite seulement, pour les patients où elle est vraiment inconnue sur toutes les visites, applique l'heuristique age - 5.7. Avec 5% de missingness, cette propagation intra-patient couvrira probablement la quasi-totalité des cas.

ledd → propagation intra-patient puis moyenne par cohorte
✅ Excellent choix, c'est la bonne stratégie.

La logique est exactement la bonne : le LEDD d'un patient ne change pas brusquement entre deux consultations, donc la moyenne des valeurs connues d'un même patient est une bien meilleure estimation que n'importe quelle valeur globale. Tomber sur la moyenne par cohorte en dernier recours est aussi une bonne idée, car l'EDA montre que ledd est corrélé à cohort (les protocoles diffèrent).

Affinage possible : plutôt que la moyenne simple des autres séances, une médiane est plus robuste si un patient a un changement de dose exceptionnel sur une visite. Mais sur 8 visites en moyenne, la différence sera minime.

Point de vigilance : avec 36.6% de missingness, il y aura des patients dont aucune visite n'a de ledd renseigné. Pour ceux-là, ta stratégie cohorte est indispensable. Vérifie bien que la moyenne cohorte est calculée uniquement sur le train et non sur le test — sinon tu introduis une fuite de données.

time_since_intake_on, time_since_intake_off, on, off
Voici plusieurs pistes, du plus simple au plus sophistiqué :

time_since_intake_on et time_since_intake_off (46% et 79% manquants)
Ces colonnes sont manquantes parce que l'examen n'a pas été fait à un moment standardisé, ou pas du tout. L'absence elle-même est un signal (→ ajouter une colonne binaire has_on_timing et has_off_timing).

Pour l'imputation de la valeur numérique :

Médiane intra-patient : si le patient a d'autres visites avec un timing renseigné, prendre la médiane. Le timing dépend du protocole du centre, pas du patient individuel, donc c'est une approximation raisonnable.
Médiane par cohorte : en fallback, comme pour ledd.
Attention : pour time_since_intake_off, 79% de NaN signifie que l'examen OFF n'a souvent tout simplement pas eu lieu. Imputer un timing sur une visite sans examen OFF est potentiellement trompeur. Réfléchis à si tu veux vraiment imputer cette valeur, ou simplement garder le NaN et laisser un arbre le gérer nativement (ex. HistGradientBoostingRegressor).
on et off (30% et 42% manquants)
C'est le cas le plus délicat. Ces colonnes sont le cœur du signal (off r=0.87 avec target). Imputer mal ici coûte cher.

Piste 1 — propagation intra-patient (temporelle) : si les visites d'un patient sont ordonnées dans le temps (via Index), tu peux interpoler lineairement entre les visites connues, ou simplement utiliser la valeur de la visite précédente (forward-fill). Cliniquement, le score d'un patient évolue lentement entre deux consultations.

Piste 2 — régression auxiliaire : entraîner un premier modèle pour prédire off à partir de on, ledd, age, time_since_intake_* et des infos patient — puis utiliser ces prédictions comme imputation. C'est de l'imputation par modèle (similaire à ce que fait sklearn.impute.IterativeImputer). C'est plus coûteux à mettre en place mais potentiellement très efficace ici vu la forte corrélation entre on et off (r=0.87).

Piste 3 (recommandée pour commencer) — indicateur de missingness + imputation simple : ajouter off_is_missing (0/1), puis imputer off par la médiane intra-patient. L'arbre apprend que "la valeur imputée n'est pas la vraie valeur" grâce à l'indicateur. C'est le meilleur ratio effort/gain pour un premier modèle.

# Modélisation : HistGradientBoostingRegressor → Skore Hub → Kaggle

Ordre d'exécution : les cellules ci-dessous se lancent **de haut en bas**, sans dépendre de l'EDA au-dessus.

1. Setup + connexion au Skore Hub
2. Données, features et CV groupée par patient (`GroupKFold`, comme le split Kaggle)
3. Pipeline : `MissingValueImputer` → suppression des ids + one-hot → `HistGradientBoostingRegressor`
4. Évaluation en CV groupée (comparée au dummy) → `put` sur le Hub
5. Rapport `EstimatorReport` sur un holdout groupé par patient → `put` → **URL à mettre dans Kaggle**
6. Fit sur tout le train → `submissions/<KEY>.csv`
7. Upload sur Kaggle

## 1. Setup et connexion au Skore Hub

In [11]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path("src").resolve()))

from parkinson.hub import load_skore_credentials
from parkinson.preprocessing import MissingValueImputer
from skore import EstimatorReport, Project, evaluate, login

# Clé du rapport sur le Hub = nom du CSV de soumission. Une NOUVELLE clé par soumission Kaggle.
KEY = "03_hgbr"

cfg = load_skore_credentials()  # lit .skore et définit la clé API
login(mode="hub")
project = Project(name="bobathon-esilv", mode="hub", workspace=cfg["workspace"])

╭───────────────────────────────── Login to Skore Hub ─────────────────────────────────╮
│                                                                                      │
│                        Successfully logged in, using API key.                        │
│                                                                                      │
╰──────────────────────────────────────────────────────────────────────────────────────╯

## 2. Données, features et CV groupée par patient

On garde `Index` et `patient_id` dans `X` : le `MissingValueImputer` a besoin de `patient_id` pour imputer patient par patient.
Le pipeline les retire ensuite, avant le modèle (sinon il apprendrait les identifiants, ce qui serait une fuite).

In [12]:
X_train = pd.read_csv("data/X_train.csv")
y_train = pd.read_csv("data/y_train.csv")
X_test = pd.read_csv("data/X_test.csv")

visits = X_train.merge(y_train, on="Index")
X = visits.drop(columns=["target"])
y = visits["target"]

from sklearn.model_selection import GroupKFold

groups = visits["patient_id"]
cv_splits = list(GroupKFold(n_splits=5).split(X, y, groups=groups))

{"X": X.shape, "X_test": X_test.shape, "n_patients": groups.nunique()}  # noqa: B018

{'X': (44590, 13), 'X_test': (11013, 13), 'n_patients': 5576}

## 3. Pipeline

L'imputation se fait **à l'intérieur** du pipeline : à chaque fold, elle n'apprend ses médianes que sur les patients du train, donc il n'y a pas de fuite vers la validation.

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

cols_to_drop = ["Index", "patient_id"]
cols_ohe = ["cohort", "gene", "rater_id"]

encoder = ColumnTransformer(
    transformers=[
        ("ids", "drop", cols_to_drop),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cols_ohe),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
)

model = make_pipeline(
    MissingValueImputer(),
    encoder,
    HistGradientBoostingRegressor(
        max_iter=100,       # Nombre maximum d'arbres
        learning_rate=0.1,  # Vitesse d'apprentissage
        random_state=42,
    ),
)
model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('missingvalueimputer', ...), ('columntransformer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,age_diag_offset,5.7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ids', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"verbose

## 4. Évaluation en CV groupée (comparée au dummy) et envoi sur le Hub

In [14]:
cv_report = evaluate(
    {"dummy": DummyRegressor(strategy="mean"), "hgbr": model},
    X,
    y,
    splitter=cv_splits,
)
cv_report.metrics.summarize()

Output()

Output()

Output()

mean_dummy     mean_hgbr     std_dummy      std_hgbr
metric                                                              
score        -8.604797e-04           NaN  5.614494e-04           NaN
r2           -8.604797e-04  7.544261e-01  5.614494e-04  7.521149e-03
rmse          1.649918e+01  8.171923e+00  3.428937e-01  2.101171e-01
mae           1.348897e+01  6.249077e+00  3.206494e-01  1.288308e-01
mape          6.549411e+14  5.926451e+13  2.115478e+14  2.089341e+13
fit_time      1.733900e-03  1.935550e+00  1.581104e-04  4.120092e-01
predict_time  2.653600e-04  1.123487e-01  6.065239e-05  5.791667e-03
Use .frame() to control the format of the output.

In [15]:

project.put(f"{KEY}_cv", cv_report.reports_["hgbr"])

Output()

d:\Sacha\projets\bobathon-esilv\.venv\Lib\site-packages\skore\_plugins\hub\artifact\envi
ronment.py:48: UserWarning: Package parkinson seems to be an editable or local install 
(loaded from outside site-packages). It will not be recorded in the inferred 
requirements.
  for requirement in infer()

d:\Sacha\projets\bobathon-esilv\.venv\Lib\site-packages\skore\_plugins\hub\artifact\envi
ronment.py:48: UserWarning: Package parkinson seems to be an editable or local install 
(loaded from outside site-packages). It will not be recorded in the inferred 
requirements.
  for requirement in infer()

d:\Sacha\projets\bobathon-esilv\.venv\Lib\site-packages\skore\_plugins\hub\artifact\envi
ronment.py:48: UserWarning: Package parkinson seems to be an editable or local install 
(loaded from outside site-packages). It will not be recorded in the inferred 
requirements.
  for requirement in infer()

d:\Sacha\projets\bobathon-esilv\.venv\Lib\site-packages\skore\_plugins\hub\artifact\envi
ronment.py:48: UserWarning: Package parkinson seems to be an editable or local install 
(loaded from outside site-packages). It will not be recorded in the inferred 
requirements.
  for requirement in infer()

d:\Sacha\projets\bobathon-esilv\.venv\Lib\site-packages\skore\_plugins\hub\artifact\envi
ronment.py:48: UserWarning: Package parkinson seems to be an editable or local install 
(loaded from outside site-packages). It will not be recorded in the inferred 
requirements.
  for requirement in infer()

Consult your report at 
]8;id=5402624;https://skore.probabl.ai/bob-slay/bobathon-esilv/cross-validations/48619\https://skore.probabl.ai/bob-slay/bobathon-esilv/cross-validations/48619]8;;\

## 5. Rapport pour la soumission Kaggle

Le README demande une URL d'**EstimatorReport**. On l'évalue sur un holdout groupé par patient (le premier fold de `cv_splits`), pour que le score reste honnête.

➡️ **Copie l'URL `https://skore.probabl.ai/…` affichée par `put`** : il faudra la coller dans la description Kaggle.

In [16]:
from sklearn.base import clone

train_idx, test_idx = cv_splits[0]
report = EstimatorReport(
    clone(model),
    X_train=X.iloc[train_idx],
    y_train=y.iloc[train_idx],
    X_test=X.iloc[test_idx],
    y_test=y.iloc[test_idx],
)
print("RMSE holdout patients :", report.metrics.rmse())
project.put(KEY, report)

Output()

RMSE holdout patients : 8.37727035029136


d:\Sacha\projets\bobathon-esilv\.venv\Lib\site-packages\skore\_plugins\hub\artifact\envi
ronment.py:48: UserWarning: Package parkinson seems to be an editable or local install 
(loaded from outside site-packages). It will not be recorded in the inferred 
requirements.
  for requirement in infer()

Consult your report at ]8;id=5402627;https://skore.probabl.ai/bob-slay/bobathon-esilv/estimators/48656\https://skore.probabl.ai/bob-slay/bobathon-esilv/estimators/48656]8;;\

## 6. Fit sur tout le train et écriture du CSV de soumission

In [17]:
final = clone(model).fit(X, y)

submission = X_test[["Index"]].copy()
submission["target"] = final.predict(X_test[X.columns])  # mêmes colonnes que l'entraînement

submissions_dir = Path("submissions")
submissions_dir.mkdir(exist_ok=True)
submission.to_csv(submissions_dir / f"{KEY}.csv", index=False)

sample = pd.read_csv("data/sample_submission.csv")
assert list(submission.columns) == ["Index", "target"]
assert len(submission) == len(sample) and submission["target"].notna().all()
submission.head()

,Index,target
0,66,18.916743
1,67,27.167747
2,68,30.840226
3,69,36.853357
4,70,30.760206
